# Task 3: Employee Attrition Classification

## Objective
Train and compare four classifiers using the Task 2 IBM HR Analytics cleaned dataset. Reuse Task 2's fixed stratified split, evaluate on its untouched test partition, tune Random Forest using training-only `GridSearchCV`, and save a reusable preprocessing-plus-model pipeline.

**Classification** predicts categories such as `Yes`/`No`; regression predicts continuous quantities. Features are the input columns; `Attrition` is the target and is excluded from transformations. High bias underfits patterns; high variance overfits training data.

Dataset attribution and setup are in the project README. The benchmark is educational, has an imbalanced target, and must not be used for punitive employment decisions.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

## 1. Load and validate data; restore Task 2 split

In [ ]:
from pathlib import Path
import pandas as pd
from src.utils import ROOT, load_dataset, load_task2_split

df = load_dataset()
train_df, test_df, split_manifest = load_task2_split(df)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Data types:\n", df.dtypes.to_string())
print("First 10 rows:\n", df.head(10).to_string(index=False))
print("Descriptive statistics:\n", df.describe(include="all").to_string())
print("Missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Target distribution:\n", df["Attrition"].value_counts())
print(f"Train/test: {len(train_df)}/{len(test_df)}; split seed={split_manifest['random_state']}; stratified by Attrition")

The fixed manifest is checked against the full dataset row ordering and target counts. No feature imputation is performed globally: the copied Task 2 preprocessing blueprint contains training-fitted imputers, outlier capping and scaling, and is cloned so folds fit their own transformations. `Attrition` remains separate.

## 2. Train four baseline models and compare their held-out metrics

In [ ]:
from src.train import train_and_evaluate

run = train_and_evaluate(ROOT, cv_folds=5)
comparison = run["table"].sort_values("f1", ascending=False).reset_index(drop=True)
comparison

The four baselines are Logistic Regression, Decision Tree, Random Forest and KNN. Positive-class precision/recall/F1 refer to `Attrition=Yes`. Accuracy is included but is not sufficient because attrition is the minority class. Higher recall reduces missed potential leavers but can lead to more false-positive outreach; higher precision limits unnecessary outreach but may miss employees who could benefit from support. These are evaluation metrics, not evidence of causal explanations.

## 3. GridSearchCV and leakage safeguards

In [ ]:
import json
results = json.loads((ROOT / "outputs/evaluation_results.json").read_text(encoding="utf-8"))
print("Training-only CV candidate F1:", results["selection_cv_f1"])
print("GridSearchCV:", results["cv"])
print("Selected using training CV only:", results["winner"])
print("Test partition rows:", results["task2_split"]["test_rows"])

GridSearchCV uses five stratified folds within Task 2's training partition and F1 scoring. Its preprocessing is inside the estimator Pipeline, so each fold fits only on that fold's training rows. The fixed test data is used solely for final evaluation, not hyperparameter or model selection. The grid covers 100/200 trees, depths 5/10/unlimited, and minimum split sizes 2/5.

## 4. Confusion matrices, ROC curves and model comparison

In [ ]:
from IPython.display import display, Image
from pathlib import Path

for path in sorted((ROOT / "images").glob("confusion_matrix_*.png")):
    print(path.name); display(Image(filename=str(path)))
for filename in ["roc_curves.png", "feature_importance.png"]:
    path = ROOT / "images" / filename
    if path.exists():
        print(filename); display(Image(filename=str(path)))
print(comparison.to_string(index=False))

The confusion matrices show false positives and false negatives at the current default classification threshold. ROC curves compare ranking performance over thresholds, but they do not set a business operating threshold. The automatic winner is selected by mean training-only CV positive-class F1; supporting held-out test metrics are shown as a final evaluation. Feature importance, if generated for a tree winner, is impurity-based and should not be interpreted causally.

## 5. Save, reload and verify unseen records

In [ ]:
import joblib
import numpy as np
model_path = ROOT / "models/best_attrition_model.pkl"
reloaded = joblib.load(model_path)
X_test = test_df.drop(columns="Attrition")
y_test = test_df["Attrition"].astype(str)
assert np.array_equal(reloaded.predict(X_test), run["model"].predict(X_test))
print("Reloaded prediction equivalence verified on", len(X_test), "test rows")
unseen = X_test.iloc[:3].copy()
if "JobRole" in unseen.columns: unseen.iloc[0, unseen.columns.get_loc("JobRole")] = "Unseen JobRole"
print("Model:", run["winner"])
print("Predictions:", reloaded.predict(unseen).tolist())
print("Probability matrix shape:", reloaded.predict_proba(unseen).shape)
print("Artifact:", model_path)

## Conclusion and limitations

See the generated comparison and `reports/Model_Evaluation_Report.pdf` for actual measured values and charts. Results are based on one held-out split from a benchmark dataset; they may not generalize to another employer or time. Check privacy, subgroup error rates, bias, drift, calibration and human oversight before any operational use. Employee attrition scores should support voluntary, equitable assistance, never automatic or punitive decisions.

In [ ]:
from src.generate_report import generate
print("Generated report:", generate())